In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [2]:
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

PATH = '/kaggle/input/competitions/playground-series-s6e10/'
train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')
orig = pd.read_csv('/kaggle/input/datasets/arseniyshutko/binary-aviation-satisfaction-129k/data.csv')

TARGET = 'satisfaction'
cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
y = train[TARGET].astype(int)
X = train.drop(columns=['id', TARGET])
X_test = test.drop(columns=['id'])
for c in cat_cols:
    cats = pd.Categorical(pd.concat([train[c], test[c], orig[c]])).categories
    X[c] = pd.Categorical(X[c], categories=cats)
    X_test[c] = pd.Categorical(X_test[c], categories=cats)
raw_cols = list(X.columns)

# 1) digit features
digit_cols = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
for df in (X, X_test):
    for c in digit_cols:
        v = df[c].fillna(0).astype('int64')
        for k in range(4):
            df[f'{c}_d{k}'] = (v // 10**k % 10).astype('int8')
freq_cols = list(X.columns)

# 2) model trained on the original data, its prediction used as a feature
orig_X = orig[raw_cols].copy()
for c in cat_cols:
    orig_X[c] = pd.Categorical(orig[c], categories=X[c].cat.categories)
orig_model = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, verbose=-1)
orig_model.fit(orig_X, orig[TARGET].astype(int))
X['orig_prob'] = orig_model.predict_proba(X[raw_cols])[:, 1]
X_test['orig_prob'] = orig_model.predict_proba(X_test[raw_cols])[:, 1]

# 3) target rates from the original data, keyed by flight distance
prior = orig[TARGET].astype(int).mean()
keys = {'fd': ['Flight Distance'], 'fd_class': ['Flight Distance', 'Class'],
        'fd_travel': ['Flight Distance', 'Type of Travel'],
        'fd_customer': ['Flight Distance', 'Customer Type'],
        'fd_age': ['Flight Distance', 'Age']}
for name, cols in keys.items():
    st = orig.groupby(cols)[TARGET].agg(['sum', 'count']).reset_index()
    st['rate'] = (st['sum'] + 2 * prior) / (st['count'] + 2)
    for src, dst in ((train, X), (test, X_test)):
        mg = src[cols].merge(st[cols + ['rate', 'count']], on=cols, how='left')
        dst[f'og_{name}_rate'] = mg['rate'].fillna(prior).values
        dst[f'og_{name}_n'] = np.log1p(mg['count'].fillna(0).values)

# 4) frequency encoding
for c in freq_cols:
    freq = pd.concat([X[c], X_test[c]]).value_counts(normalize=True)
    X[f'{c}_freq'] = X[c].map(freq).astype('float32')
    X_test[f'{c}_freq'] = X_test[c].map(freq).astype('float32')

# 5) route profile (same flight distance = same route)
n = len(X)
both = pd.concat([X[raw_cols], X_test[raw_cols]], ignore_index=True)
for c in cat_cols:
    both[c] = both[c].cat.codes
g = both.groupby('Flight Distance')
cnt = g['Flight Distance'].transform('count').values
X['route_count'], X_test['route_count'] = cnt[:n], cnt[n:]
for c in [c for c in raw_cols if c != 'Flight Distance']:
    m = g[c].transform('mean').values
    d = both[c].values - m
    X[f'route_m_{c}'], X_test[f'route_m_{c}'] = m[:n], m[n:]
    X[f'route_d_{c}'], X_test[f'route_d_{c}'] = d[:n], d[n:]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X.shape, X_test.shape)

(699635, 126) (299844, 126)


In [3]:
import warnings
warnings.filterwarnings('ignore')
X, X_test = X.copy(), X_test.copy()   # defragment

yv = y.values
all_raw = pd.concat([train.drop(columns=['id', TARGET]), test.drop(columns=['id'])], ignore_index=True)

def make_key(cols):
    return all_raw.groupby(cols, sort=False, dropna=False).ngroup().values

te_keys = {
    'fd':          make_key(['Flight Distance']),
    'fd_class':    make_key(['Flight Distance', 'Class']),
    'fd_travel':   make_key(['Flight Distance', 'Type of Travel']),
    'fd_customer': make_key(['Flight Distance', 'Customer Type']),
    'fd_age':      make_key(['Flight Distance', 'Age']),
    'age':         make_key(['Age']),
    'age_class':   make_key(['Age', 'Class']),
    'age_travel':  make_key(['Age', 'Type of Travel']),
}

def smoothed(k_fit, y_fit, k_apply, m=20):
    prior = y_fit.mean()
    st = pd.DataFrame({'k': k_fit, 'y': y_fit}).groupby('k')['y'].agg(['sum', 'count'])
    s = pd.Series(k_apply)
    return ((s.map(st['sum']).fillna(0) + m * prior) / (s.map(st['count']).fillna(0) + m)).values

es_mask = np.random.default_rng(2027).random(n) < 0.10

# target encoding - tried it, didn't help CV, not used below
def run_cv2(Xa, Xt, name, use_te=False):
    oof = np.zeros(n); pred = np.zeros(len(Xt))
    for fold, (tr, va) in enumerate(skf.split(Xa, y)):
        fit, es = tr[~es_mask[tr]], tr[es_mask[tr]]
        Xfit, Xes, Xva, Xte = Xa.iloc[fit].copy(), Xa.iloc[es].copy(), Xa.iloc[va].copy(), Xt.copy()
        if use_te:
            inner = np.random.default_rng(fold).permutation(len(fit)) % 5
            for kn, k in te_keys.items():
                ktr, kte = k[:n], k[n:]
                col = np.zeros(len(fit))
                for j in range(5):
                    a, b = fit[inner != j], inner == j
                    col[b] = smoothed(ktr[a], yv[a], ktr[fit[b]])
                Xfit[f'te_{kn}'] = col
                Xes[f'te_{kn}'] = smoothed(ktr[fit], yv[fit], ktr[es])
                Xva[f'te_{kn}'] = smoothed(ktr[fit], yv[fit], ktr[va])
                Xte[f'te_{kn}'] = smoothed(ktr[fit], yv[fit], kte)
        model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, verbose=-1)
        model.fit(Xfit, yv[fit], eval_set=[(Xes, yv[es])], eval_metric='auc',
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = model.predict_proba(Xva)[:, 1]
        pred += model.predict_proba(Xte)[:, 1] / 5
        print(f'Fold {fold}: AUC = {roc_auc_score(yv[va], oof[va]):.5f}')
    print('CV AUC:', roc_auc_score(yv, oof))
    pd.DataFrame({'id': test['id'], 'satisfaction': pred}).to_csv(f'submission_{name}.csv', index=False)
    return oof, pred

In [4]:
def run_cv3(Xa, Xt, name, params):
    oof = np.zeros(n); pred = np.zeros(len(Xt))
    for fold, (tr, va) in enumerate(skf.split(Xa, y)):
        fit, es = tr[~es_mask[tr]], tr[es_mask[tr]]
        model = lgb.LGBMClassifier(**params)
        model.fit(Xa.iloc[fit], yv[fit], eval_set=[(Xa.iloc[es], yv[es])], eval_metric='auc',
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = model.predict_proba(Xa.iloc[va])[:, 1]
        pred += model.predict_proba(Xt)[:, 1] / 5
        print(f'Fold {fold}: AUC = {roc_auc_score(yv[va], oof[va]):.5f}  iters = {model.best_iteration_}')
    print('CV AUC:', roc_auc_score(yv, oof))
    pd.DataFrame({'id': test['id'], 'satisfaction': pred}).to_csv(f'submission_{name}.csv', index=False)
    return oof, pred

params = dict(
    n_estimators=3000, learning_rate=0.05,
    num_leaves=63, max_depth=8, min_child_samples=5,
    reg_lambda=5.0, reg_alpha=0.05,
    subsample=0.9, subsample_freq=1, colsample_bytree=0.9,
    verbose=-1,
)

In [5]:
params_lr = dict(params, learning_rate=0.02, n_estimators=5000)
oof_lr, pred_lr = run_cv3(X, X_test, 'tuned_lr02', params_lr)
np.save('oof_lgb.npy', oof_lr)
np.save('pred_lgb.npy', pred_lr)

Fold 0: AUC = 0.96077  iters = 825
Fold 1: AUC = 0.95982  iters = 691
Fold 2: AUC = 0.96116  iters = 583
Fold 3: AUC = 0.96074  iters = 640
Fold 4: AUC = 0.96049  iters = 670
CV AUC: 0.9605888188699134


In [6]:
import xgboost as xgb

def run_xgb(Xa, Xt, name, params):
    oof = np.zeros(n); pred = np.zeros(len(Xt))
    for fold, (tr, va) in enumerate(skf.split(Xa, y)):
        fit, es = tr[~es_mask[tr]], tr[es_mask[tr]]
        model = xgb.XGBClassifier(**params)
        model.fit(Xa.iloc[fit], yv[fit], eval_set=[(Xa.iloc[es], yv[es])], verbose=False)
        oof[va] = model.predict_proba(Xa.iloc[va])[:, 1]
        pred += model.predict_proba(Xt)[:, 1] / 5
        print(f'Fold {fold}: AUC = {roc_auc_score(yv[va], oof[va]):.5f}  iters = {model.best_iteration}')
    print('CV AUC:', roc_auc_score(yv, oof))
    pd.DataFrame({'id': test['id'], 'satisfaction': pred}).to_csv(f'submission_{name}.csv', index=False)
    np.save(f'oof_{name}.npy', oof); np.save(f'pred_{name}.npy', pred)
    return oof, pred

xgb_params = dict(
    n_estimators=5000, learning_rate=0.04, max_depth=7, min_child_weight=3.6,
    subsample=0.94, colsample_bytree=0.4, colsample_bynode=0.45,
    reg_lambda=33.0, reg_alpha=17.0, max_bin=1024,
    tree_method='hist', device='cuda', enable_categorical=True,
    eval_metric='auc', early_stopping_rounds=150,
)
oof_xgb, pred_xgb = run_xgb(X, X_test, 'xgb', xgb_params)

Fold 0: AUC = 0.96075  iters = 891
Fold 1: AUC = 0.95976  iters = 1039
Fold 2: AUC = 0.96122  iters = 747
Fold 3: AUC = 0.96077  iters = 519
Fold 4: AUC = 0.96065  iters = 563
CV AUC: 0.9606157556341686


In [7]:
from scipy.stats import rankdata

oof_lgb, pred_lgb = np.load('oof_lgb.npy'), np.load('pred_lgb.npy')
oof_xgb, pred_xgb = np.load('oof_xgb.npy'), np.load('pred_xgb.npy')

def rk(a):
    return rankdata(a) / len(a)

print('LGB  :', roc_auc_score(yv, oof_lgb))
print('XGB  :', roc_auc_score(yv, oof_xgb))
print('corr :', np.corrcoef(oof_lgb, oof_xgb)[0, 1])

best_w, best_auc = 0, 0
for w in np.arange(0, 1.01, 0.1):
    auc = roc_auc_score(yv, w * rk(oof_lgb) + (1 - w) * rk(oof_xgb))
    print(f'w_lgb = {w:.1f}  AUC = {auc:.5f}')
    if auc > best_auc:
        best_w, best_auc = w, auc

print(f'\nbest w_lgb = {best_w:.1f}  CV = {best_auc:.5f}')
blend = best_w * rk(pred_lgb) + (1 - best_w) * rk(pred_xgb)
pd.DataFrame({'id': test['id'], 'satisfaction': blend}).to_csv('submission_blend.csv', index=False)

LGB  : 0.9605888188699134
XGB  : 0.9606157556341686
corr : 0.9990377983505945
w_lgb = 0.0  AUC = 0.96062
w_lgb = 0.1  AUC = 0.96067
w_lgb = 0.2  AUC = 0.96072
w_lgb = 0.3  AUC = 0.96075
w_lgb = 0.4  AUC = 0.96077
w_lgb = 0.5  AUC = 0.96077
w_lgb = 0.6  AUC = 0.96076
w_lgb = 0.7  AUC = 0.96074
w_lgb = 0.8  AUC = 0.96070
w_lgb = 0.9  AUC = 0.96065
w_lgb = 1.0  AUC = 0.96059

best w_lgb = 0.5  CV = 0.96077


In [8]:
from catboost import CatBoostClassifier

def run_cb(Xa, Xt, name, params):
    Xa, Xt = Xa.copy(), Xt.copy()
    for c in cat_cols:
        Xa[c] = Xa[c].astype(str)
        Xt[c] = Xt[c].astype(str)
    oof = np.zeros(n); pred = np.zeros(len(Xt))
    for fold, (tr, va) in enumerate(skf.split(Xa, y)):
        fit, es = tr[~es_mask[tr]], tr[es_mask[tr]]
        model = CatBoostClassifier(**params, cat_features=cat_cols)
        model.fit(Xa.iloc[fit], yv[fit], eval_set=(Xa.iloc[es], yv[es]))
        oof[va] = model.predict_proba(Xa.iloc[va])[:, 1]
        pred += model.predict_proba(Xt)[:, 1] / 5
        print(f'Fold {fold}: AUC = {roc_auc_score(yv[va], oof[va]):.5f}  iters = {model.get_best_iteration()}')
    print('CV AUC:', roc_auc_score(yv, oof))
    np.save(f'oof_{name}.npy', oof); np.save(f'pred_{name}.npy', pred)
    return oof, pred

cb_params = dict(
    iterations=10000, learning_rate=0.05, depth=8, l2_leaf_reg=3,
    border_count=254, task_type='GPU', random_seed=42,
    od_type='Iter', od_wait=200, verbose=0,
)
oof_cb, pred_cb = run_cb(X, X_test, 'cb', cb_params)

Fold 0: AUC = 0.96041  iters = 1089
Fold 1: AUC = 0.95963  iters = 1192
Fold 2: AUC = 0.96104  iters = 783
Fold 3: AUC = 0.96050  iters = 1384
Fold 4: AUC = 0.96023  iters = 1022
CV AUC: 0.9603504079651299


In [9]:
oof_cb, pred_cb = np.load('oof_cb.npy'), np.load('pred_cb.npy')

oofs  = {'lgb': rk(oof_lgb),  'xgb': rk(oof_xgb),  'cb': rk(oof_cb)}
preds = {'lgb': rk(pred_lgb), 'xgb': rk(pred_xgb), 'cb': rk(pred_cb)}

for k in oofs:
    print(f'{k:4s}: {roc_auc_score(yv, oofs[k]):.5f}')
print('corr lgb-cb:', np.corrcoef(oof_lgb, oof_cb)[0, 1])
print('corr xgb-cb:', np.corrcoef(oof_xgb, oof_cb)[0, 1])

best = (0, None)
for a in np.arange(0, 1.01, 0.1):
    for b in np.arange(0, 1.01 - a, 0.1):
        c = 1 - a - b
        auc = roc_auc_score(yv, a * oofs['lgb'] + b * oofs['xgb'] + c * oofs['cb'])
        if auc > best[0]:
            best = (auc, (a, b, c))

auc, (a, b, c) = best
print(f'\nbest: lgb={a:.1f} xgb={b:.1f} cb={c:.1f}  CV = {auc:.5f}')
blend3 = a * preds['lgb'] + b * preds['xgb'] + c * preds['cb']
pd.DataFrame({'id': test['id'], 'satisfaction': blend3}).to_csv('submission_blend3.csv', index=False)

lgb : 0.96059
xgb : 0.96062
cb  : 0.96035
corr lgb-cb: 0.9991874386325477
corr xgb-cb: 0.9989257930387763

best: lgb=0.4 xgb=0.4 cb=0.2  CV = 0.96079


In [ ]:
for root, dirs, files in os.walk('/kaggle/input/datasets/cdeotte'):
    for f in sorted(files):
        print(os.path.join(root, f))

In [11]:
TFM = '/kaggle/input/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions/'
names = ['tabpfn35', 'kumolarge', 'kumosmall', 'tabfm', 'exaone', 'limix2',
         'causilo', 'tabicl2', 'mitra2', 'realmlp', 'xgboost']

ext_oof, ext_pred = {}, {}
for m in names:
    o, t = np.load(TFM + f'oof_{m}.npy'), np.load(TFM + f'test_{m}.npy')
    if o.ndim == 2:
        o = o[:, -1]
    if t.ndim == 2:
        t = t[:, -1]
    ext_oof[m], ext_pred[m] = o, t
    print(f'{m:10s} oof {o.shape} test {t.shape}  '
          f'AUC = {roc_auc_score(yv, o):.6f}  corr_lgb = {np.corrcoef(oof_lgb, o)[0, 1]:.4f}')

tabpfn35   oof (699635,) test (299844,)  AUC = 0.961145  corr_lgb = 0.9964
kumolarge  oof (699635,) test (299844,)  AUC = 0.960036  corr_lgb = 0.9958
kumosmall  oof (699635,) test (299844,)  AUC = 0.958484  corr_lgb = 0.9952
tabfm      oof (699635,) test (299844,)  AUC = 0.958871  corr_lgb = 0.9952
exaone     oof (699635,) test (299844,)  AUC = 0.958557  corr_lgb = 0.9955
limix2     oof (699635,) test (299844,)  AUC = 0.958474  corr_lgb = 0.9952
causilo    oof (699635,) test (299844,)  AUC = 0.958275  corr_lgb = 0.9955
tabicl2    oof (699635,) test (299844,)  AUC = 0.958088  corr_lgb = 0.9945
mitra2     oof (699635,) test (299844,)  AUC = 0.957075  corr_lgb = 0.9943
realmlp    oof (699635,) test (299844,)  AUC = 0.959311  corr_lgb = 0.9958
xgboost    oof (699635,) test (299844,)  AUC = 0.959852  corr_lgb = 0.9959


In [12]:
pool = {'lgb': rk(oof_lgb), 'xgb': rk(oof_xgb), 'cb': rk(oof_cb)}
pool_t = {'lgb': rk(pred_lgb), 'xgb': rk(pred_xgb), 'cb': rk(pred_cb)}
for m in names:
    pool[f'ext_{m}'] = rk(ext_oof[m])
    pool_t[f'ext_{m}'] = rk(ext_pred[m])

chosen, cur, cur_auc = [], None, 0
for step in range(30):
    best_m, best_auc = None, cur_auc
    for m, v in pool.items():
        cand = v if cur is None else (cur * len(chosen) + v) / (len(chosen) + 1)
        auc = roc_auc_score(yv, cand)
        if auc > best_auc:
            best_m, best_auc = m, auc
    if best_m is None:
        break
    cur = pool[best_m] if cur is None else (cur * len(chosen) + pool[best_m]) / (len(chosen) + 1)
    chosen.append(best_m)
    cur_auc = best_auc
    print(f'step {step:2d}: + {best_m:15s} CV = {cur_auc:.5f}')

from collections import Counter
w = Counter(chosen)
print('\nweights:', {k: f'{v / len(chosen):.2f}' for k, v in w.most_common()})
print(f'final CV = {cur_auc:.5f}')

blend_hc = sum(pool_t[k] * v for k, v in w.items()) / len(chosen)
pd.DataFrame({'id': test['id'], 'satisfaction': blend_hc}).to_csv('submission_hc.csv', index=False)

step  0: + ext_tabpfn35    CV = 0.96115
step  1: + lgb             CV = 0.96141
step  2: + ext_tabpfn35    CV = 0.96144
step  3: + xgb             CV = 0.96145
step  4: + ext_tabpfn35    CV = 0.96147

weights: {'ext_tabpfn35': '0.60', 'lgb': '0.20', 'xgb': '0.20'}
final CV = 0.96147
